# Simple Classification: Classical vs Quantum Exercise

In this assignment you will compare two models on the same binary classification problem:

1. a small classical neural network implemented in PyTorch,
2. a hybrid quantum-classical model implemented with PennyLane and PyTorch.

The goal is not to prove quantum advantage. The goal is to understand the workflow:

$$
\text{data}
\rightarrow
\text{classical baseline}
\rightarrow
\text{quantum layer}
\rightarrow
\text{comparison of accuracy, loss, and runtime}.
$$

Use the existing project environment:

```bash
uv run jupyter lab
```

The required packages are already in the root `pyproject.toml`: `numpy`, `matplotlib`, `scikit-learn`, `torch`, and `pennylane`.


## What You Should Achieve

By the end of the exercise, your notebook should contain:

- a plotted binary dataset,
- a trained classical classifier,
- a trained hybrid quantum-classical classifier,
- loss curves for both models,
- train/test accuracy for both models,
- a short comparison of runtime and model behavior.

Keep the comparison fair: both models should use the same dataset split and the same evaluation metric.

Suggested target: both models should learn the nonlinear circle dataset noticeably better than random guessing.


## Imports

This notebook intentionally avoids `pip install` cells. Use the repository's `uv` environment instead.


In [ ]:
# Pseudo-code:
#   1. import numerical, plotting, dataset, torch, and PennyLane tools
#   2. set random seeds
#   3. select torch device

# Setup: imports and reproducibility.
import time

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import torch
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cpu")
print("PennyLane version:", qml.__version__)
print("PyTorch version:", torch.__version__)
print("Torch device:", device)


## A Tiny PennyLane (`qml`) Example

PennyLane is imported as `qml`. A quantum model usually has three pieces:

1. a device,
2. a quantum node (`QNode`),
3. measurements returned as numerical values.

The example below is not the assignment solution. It only shows the syntax you will use later.


In [ ]:
# Pseudo-code:
#   1. create a one-qubit simulator
#   2. define a QNode with one input angle
#   3. measure a Pauli-Z expectation value
#   4. print the circuit and output

# Minimal PennyLane example before building the assignment model.
import pennylane as qml
import torch

demo_dev = qml.device("default.qubit", wires=1)

@qml.qnode(demo_dev, interface="torch")
def one_qubit_demo(angle):
    qml.RY(angle, wires=0)
    return qml.expval(qml.PauliZ(0))

demo_angle = torch.tensor(0.7)
demo_output = one_qubit_demo(demo_angle)

print(qml.draw(one_qubit_demo)(demo_angle))
print("output <Z>:", float(demo_output))


## Dataset

Use a noisy two-circle dataset. It is intentionally nonlinear: a single straight line cannot separate the two classes well.

The labels are binary:

$$
y \in \{0,1\}.
$$

The input has two features:

$$
x = (x_1, x_2).
$$


In [ ]:
# Pseudo-code:
#   1. generate the circle dataset
#   2. split into train and test sets
#   3. convert arrays to torch tensors
#   4. visualize the training data

# Data setup shared by the classical and quantum models.
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

X, y = make_circles(
    n_samples=600,
    noise=0.08,
    factor=0.45,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train[:, None], dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test[:, None], dtype=torch.float32)

plt.figure(figsize=(5.2, 4.4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=28, alpha=0.85)
plt.title("Training data: noisy circles")
plt.xlabel("x1")
plt.ylabel("x2")
plt.axis("equal")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("train shape:", X_train_t.shape, y_train_t.shape)
print("test shape:", X_test_t.shape, y_test_t.shape)


## Task 1 - Classical Baseline

Implement a small PyTorch neural network for binary classification.

Requirements:

- input dimension: 2,
- output dimension: 1 logit,
- use `torch.nn.BCEWithLogitsLoss`,
- do **not** put a sigmoid inside the model if you use `BCEWithLogitsLoss`,
- train on `X_train_t`, `y_train_t`,
- evaluate on `X_test_t`, `y_test_t`.

Suggested architecture:

$$
(x_1,x_2)
\rightarrow
\text{Linear}
\rightarrow
\text{nonlinearity}
\rightarrow
\text{Linear}
\rightarrow
\text{logit}.
$$


In [ ]:
# TODO: implement the classical classifier.
#
# Suggested structure:
#
# class ClassicalNet(torch.nn.Module):
#     def __init__(self):
#         super().__init__()
#         ...
#
#     def forward(self, x):
#         ...
#         return logits
#
# classical_model = ClassicalNet()


## Task 2 - Hybrid Quantum-Classical Model

Build a hybrid model using a PennyLane quantum layer.

A typical structure is:

$$
x
\rightarrow
\text{classical preprocessing}
\rightarrow
\text{quantum circuit}
\rightarrow
\text{classical postprocessing}
\rightarrow
\text{logit}.
$$

Minimum requirements:

- use at least 2 qubits,
- encode the two input features with rotation gates,
- include at least one trainable quantum layer,
- measure expectation values,
- connect the quantum circuit to PyTorch using `qml.qnn.TorchLayer`,
- return one logit for binary classification.

Keep the quantum circuit shallow. The goal is to understand the interface, not to build a large quantum model.


In [ ]:
# TODO: implement the quantum circuit and wrap it as a torch layer.
#
# Hints:
#
# n_qubits = 2
# q_depth = 1
# dev = qml.device("default.qubit", wires=n_qubits)
#
# @qml.qnode(dev, interface="torch")
# def qnode(inputs, weights):
#     # 1. encode inputs using rotations
#     # 2. apply trainable rotations and/or entangling gates
#     # 3. return expectation values
#     ...
#
# weight_shapes = {"weights": (...)}
# qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)


In [ ]:
# TODO: implement the hybrid model.
#
# Suggested structure:
#
# class HybridQuantumNet(torch.nn.Module):
#     def __init__(self):
#         super().__init__()
#         ...
#
#     def forward(self, x):
#         ...
#         return logits
#
# quantum_model = HybridQuantumNet()


## Task 3 - Training Loop

Write one training function that can train either model.

Recommended ingredients:

- `torch.nn.BCEWithLogitsLoss()`,
- `torch.optim.Adam`,
- a fixed number of epochs,
- loss history,
- accuracy computed from `torch.sigmoid(logits) >= 0.5`.

Keep the dataset small enough that the quantum model runs in reasonable time.


In [ ]:
# TODO: implement reusable training and evaluation helpers.
#
# Suggested functions:
#
# def binary_accuracy_from_logits(logits, targets):
#     ...
#
# def train_model(model, X_train, y_train, X_test, y_test, epochs, lr):
#     ...
#     return history


## Task 4 - Comparison

Train both models and compare them.

Report:

| Model | Train accuracy | Test accuracy | Runtime | Notes |
|---|---:|---:|---:|---|
| Classical NN |  |  |  |  |
| Hybrid quantum NN |  |  |  |  |

Include at least one plot:

- loss curves for both models, or
- decision regions for both models, or
- predicted labels on the test set.

In your written comparison, answer:

1. Which model trained faster?
2. Which model performed better on the test set?
3. What changed when you introduced the quantum layer?
4. Did the quantum model justify its extra cost on this small task?


In [ ]:
# TODO: train both models and fill in the comparison table.
#
# Keep this section concise:
# - train classical_model
# - train quantum_model
# - plot loss curves or predictions
# - print train/test accuracy and runtime


## Submission Checklist

Before submitting, make sure your notebook contains:

- the generated dataset plot,
- your classical model implementation,
- your quantum model implementation,
- training curves or prediction plots,
- final train/test metrics,
- a short written comparison.

Do not only report accuracy. Discuss runtime and implementation complexity as well.
